# **Movie_Recommendation_System**

**Content-Filtering**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import zipfile
zip=zipfile.ZipFile('/content/drive/MyDrive/Data Science/Project/TMBD_dataset.zip')
zip.extractall()
zip.close()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Data Science/Project/TMBD_dataset.zip'

In [ ]:
movie=pd.read_csv('/content/tmdb_5000_movies.csv')


In [ ]:
credits=pd.read_csv('/content/tmdb_5000_credits.csv')

In [ ]:
movie.head(1)

In [ ]:
movie.columns

In [ ]:
movie= movie[['title','keywords','genres','overview']]

In [ ]:
movie.head()

In [ ]:
credits.head(2)

In [ ]:
df=pd.merge(movie,credits,left_on='title',right_on='title')

In [ ]:
df.head()

In [ ]:
def convert(obj):
  L=[]
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L

In [ ]:
df['genres']=df['genres'].apply(convert)

In [ ]:
df['genres']

In [ ]:
df['keywords']=df['keywords'].apply(convert)

In [ ]:
df['keywords'][0]

In [ ]:
df['cast'][0]

In [ ]:
df['cast']=df['cast'].apply(lambda  x:[i['name'] for i in ast.literal_eval(x)[:3]]) # only top 3 actor

In [ ]:
df['cast']

In [ ]:
df['crew'][0]

In [ ]:
df['crew']=df['crew'].apply(lambda  x:[i['name'] for i in ast.literal_eval(x) if i['job'] == 'Director'] )

In [ ]:
df['crew']

In [ ]:
df['tags']=df['genres']+df['keywords']+df['cast']+df['crew']

In [ ]:
df['tags']

In [ ]:
df.columns

In [ ]:
df=df[['movie_id','title','overview','tags']]

In [ ]:
df.head()

In [ ]:
df['tags']=df['tags'].apply(lambda x: " ".join(x))

In [ ]:
df['tags']=df['tags'].apply(lambda x: x.lower())

In [ ]:
df['tags']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(stop_words='english')
tfidf_matrix=tfidf.fit_transform(df['tags'])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim=cosine_similarity(tfidf_matrix,tfidf_matrix)

In [ ]:
def get_recommendation(title,cosine_sim=cosine_sim):
  idx=df[df['title']==title].index[0]
  sim_scores=list(enumerate(cosine_sim[idx]))
  sim_scores=sorted(sim_scores,key=lambda x: x[1],reverse=True)
  sim_scores=sim_scores[1:11]
  movie_indices=[i[0] for i in sim_scores]
  return df['title'].iloc[movie_indices]

In [ ]:
print(get_recommendation('Avatar'))

In [ ]:
import pickle
with open('movie_data.pkl','wb')as file:
  pickle.dump(df,cosine_sim,file)
